# Conferência do motor de frete

Ferramenta do relatório em `RELATORIO.md`. Não usa banco e não chama a API.

Começa com um **exemplo manual** (entrada → esperado → `calcular()` → comparação).
Depois roda os 21 casos de `casos.json`.

- casos com `modo: faixa` → `preco_da_faixa()`
- casos com `modo: calculo` → `calcular()`


## 0. Preparar o ambiente


In [1]:
from pathlib import Path
import sys
import json
from datetime import date, datetime
from decimal import Decimal

PASTA = Path.cwd()
while not (PASTA / "src" / "motor_frete").exists():
    if PASTA.parent == PASTA:
        raise RuntimeError("não achei src/motor_frete — abra o notebook a partir do repo")
    PASTA = PASTA.parent

NOTEBOOK_DIR = PASTA / "validacao-motor-frete"
sys.path.insert(0, str(PASTA / "src"))

from motor_frete import generalidades as ger
from motor_frete.calculo import calcular
from motor_frete.esquemas import Embarque, MotivoDescarte
from motor_frete.resolucao import Contexto, DescarteError, Faixas, preco_da_faixa

def money(v) -> Decimal:
    return Decimal(str(v))

def brl(v) -> str:
    return f"{money(v):.2f}"

quando = date(2026, 10, 1)
tolerancia = Decimal("0.01")

print(f"repo:     {PASTA}")
print(f"notebook: {NOTEBOOK_DIR}")
print(f"pronto para calcular · data {quando}")


repo:     /Users/guilhermeoliveira/Documents/git_repos/motor-frete
notebook: /Users/guilhermeoliveira/Documents/git_repos/motor-frete/validacao-motor-frete
pronto para calcular · data 2026-10-01


## 1. Exemplo manual — entrada, esperado, chamada, comparação

O padrão de toda a validação, num caso só, escrito na célula:

1. montar a **entrada**
2. declarar o **esperado** (conta na mão, antes de olhar o motor)
3. chamar `calcular()`
4. **comparar** campo a campo

Conta deste exemplo:

- frete-peso = R$ 250
- GRIS = `maximo(5, nota × 0,002)` com nota R$ 10.000 → R$ 20
- líquido = 250 + 20 = 270
- ICMS = 270 × 0,12 = R$ 32,40
- **total esperado = R$ 302,40**


In [2]:
# ── 1. Entrada (o que o motor recebe) ──────────────────────────────────
entrada = {
    "embarque": {
        "origem_cep": "60000000",
        "destino_cep": "20000000",
        "peso_bruto_kg": "100",
        "volume_m3": "0",
        "valor_nf": "10000",       # nota de R$ 10.000 — base do GRIS
        "quantidade_volumes": 4,
    },
    "tarifa": {
        "peso_taxado": "100",
        "preco_frete_peso": "250",  # faixa já fechada em reais
        "icms": "0.12",             # 12%
        "tarcli": "AD",
        "uf_destino": "RJ",
        "formulas": {
            "gris": "maximo(5,[fv]*0.002)",
            "tst": "[gris]",
        },
    },
}

# ── 2. Esperado (conta feita na mão, ANTES de olhar o motor) ───────────
# VN2 = 250
# GRIS = max(5, 10000*0.002) = max(5, 20) = 20
# líquido = 250 + 20 = 270
# ICMS = 270 * 0.12 = 32.40
# total = 270 + 32.40 = 302.40
esperado = {
    "frete_peso": Decimal("250.00"),
    "generalidades": Decimal("20.00"),
    "icms": Decimal("32.40"),
    "frete_liquido": Decimal("270.00"),
    "frete_total": Decimal("302.40"),
}

# ── 3. Chamar a função ─────────────────────────────────────────────────
emb = entrada["embarque"]
tar = entrada["tarifa"]
embarque = Embarque(
    referencia="exemplo-manual",
    origem_cep=emb["origem_cep"],
    destino_cep=emb["destino_cep"],
    peso_bruto_kg=money(emb["peso_bruto_kg"]),
    volume_m3=money(emb["volume_m3"]),
    valor_nf=money(emb["valor_nf"]),
    quantidade_volumes=emb["quantidade_volumes"],
)
perfil = ger.Perfil.de_linha("exemplo-manual", tar["formulas"])
ctx = Contexto(
    transportadora_id=7,
    transportadora="JAMEF",
    regiao_atendida_id=55,
    praca_id=10,
    praca="CAPITAL",
    modalidade_id=1,
    modalidade="RODOVIARIO",
    fator_cubagem=money("300"),
    uf_destino=tar["uf_destino"],
    tabela_frete_id=201,
    nome_tabela="exemplo",
    tabela_frete_peso_id=1,
    faixa_valor_id=1,
    peso_taxado=money(tar["peso_taxado"]),
    preco_frete_peso=money(tar["preco_frete_peso"]),
    icms=money(tar["icms"]),
    tarcli=tar["tarcli"],
    prazo_dias=3,
    perfil=perfil,
)
cotacao = calcular(embarque, ctx, quando)

# ── 4. Comparar esperado × motor ───────────────────────────────────────
print("campo              esperado    motor       ok?")
print("-" * 50)
tudo_ok = True
for campo, valor_esperado in esperado.items():
    obtido = money(getattr(cotacao, campo))
    bateu = abs(valor_esperado - obtido) <= Decimal("0.01")
    tudo_ok = tudo_ok and bateu
    marca = "ok" if bateu else "FALHA"
    print(f"{campo:<18} {brl(valor_esperado):>10}  {brl(obtido):>10}  {marca}")

print()
print("PASSOU" if tudo_ok else "DIVERGIU", "— total do motor:", brl(cotacao.frete_total))


campo              esperado    motor       ok?
--------------------------------------------------
frete_peso             250.00      250.00  ok
generalidades           20.00       20.00  ok
icms                    32.40       32.40  ok
frete_liquido          270.00      270.00  ok
frete_total            302.40      302.40  ok

PASSOU — total do motor: 302.40


## 2. Carregar os 21 casos


In [3]:
CASOS_PATH = NOTEBOOK_DIR / "casos.json"
SAIDA_DIR = NOTEBOOK_DIR / "saida"
SAIDA_PATH = SAIDA_DIR / "conferencia.json"

dados = json.loads(CASOS_PATH.read_text(encoding="utf-8"))
quando = date.fromisoformat(dados["data_referencia"])
tolerancia = money(dados["tolerancia"])
casos = dados["casos"]

print(f"{len(casos)} casos · data {quando} · tolerância {tolerancia}")
for c in casos:
    print(f"  caso {c.get('n', '?')}  {c['id']:<28} {c['titulo']}")


21 casos · data 2026-10-01 · tolerância 0.01
  caso 1  faixa_dentro_7kg             Peso dentro da faixa — banda 0–10 kg
  caso 2  faixa_limite_70kg            Peso no limite da banda — 70 kg
  caso 3  excedente_acima_do_teto      Peso acima do teto, com excedente
  caso 4  peso_fora_sem_excedente      Peso acima do teto, sem excedente
  caso 5  so_frete_peso                Só o frete-peso
  caso 6  minimo_do_frete_pega         O mínimo do frete-peso pega
  caso 7  minimo_do_frete_nao_pega     O mínimo do frete-peso não pega
  caso 8  gris_percentual              GRIS — o percentual ganha do piso
  caso 9  gris_piso                    GRIS — o piso ganha do percentual
  caso 10  gris_e_despacho              GRIS e despacho em linhas separadas
  caso 11  tarcli_liga                  tarcli liga a cobrança de pallet
  caso 12  tarcli_desliga               tarcli desliga a cobrança de pallet
  caso 13  pallet_informado             Pallet com a quantidade certa
  caso 14  pallet_estimado  

## 3. Helpers dos dois modos (faixa e cálculo)


In [4]:
def montar_faixas(raw: dict) -> Faixas:
    bandas = tuple(
        {
            "id": int(b["id"]),
            "peso_inicio": money(b["peso_inicio"]),
            "peso_fim": None if b.get("peso_fim") is None else money(b["peso_fim"]),
            "minimo": None if b.get("minimo") is None else money(b["minimo"]),
            "valor": None if b.get("valor") is None else money(b["valor"]),
            "tipo_calculo": b.get("tipo_calculo", "Faixa"),
        }
        for b in (raw.get("bandas") or [])
    )
    excedente = None
    if raw.get("excedente"):
        e = raw["excedente"]
        excedente = {
            "id": int(e["id"]),
            "peso_inicio": money(e["peso_inicio"]),
            "peso_fim": None if e.get("peso_fim") is None else money(e["peso_fim"]),
            "minimo": None if e.get("minimo") is None else money(e["minimo"]),
            "valor": None if e.get("valor") is None else money(e["valor"]),
            "tipo_calculo": e.get("tipo_calculo", "Excedente"),
        }
    exc_min = money(raw["excedente_minimo"]) if raw.get("excedente_minimo") is not None else None
    return Faixas(bandas=bandas, excedente=excedente, excedente_minimo=exc_min)


def rodar_faixa(caso: dict) -> dict:
    faixas = montar_faixas(caso["faixas"])
    peso = money(caso["peso"])
    try:
        preco, faixa_id, com_exc = preco_da_faixa(faixas, peso)
        return {
            "ok": True,
            "preco": brl(preco),
            "faixa_id": faixa_id,
            "aplicou_excedente": com_exc,
            "motivo": None,
            "detalhe": None,
        }
    except DescarteError as e:
        return {
            "ok": False,
            "preco": None,
            "faixa_id": None,
            "aplicou_excedente": False,
            "motivo": e.motivo.value if isinstance(e.motivo, MotivoDescarte) else str(e.motivo),
            "detalhe": e.detalhe,
        }


def rodar_calculo(caso: dict):
    emb = caso["embarque"]
    tar = caso["tarifa"]
    embarque = Embarque(
        referencia=caso["id"],
        origem_cep=emb["origem_cep"],
        destino_cep=emb["destino_cep"],
        peso_bruto_kg=money(emb["peso_bruto_kg"]),
        volume_m3=money(emb.get("volume_m3", "0")),
        valor_nf=money(emb["valor_nf"]),
        quantidade_volumes=int(emb.get("quantidade_volumes", 1)),
        quantidade_pallets=emb.get("quantidade_pallets"),
    )
    formulas = tar.get("formulas") or None
    perfil = ger.Perfil.de_linha(caso["id"], formulas) if formulas else None
    ctx = Contexto(
        transportadora_id=7,
        transportadora=tar.get("transportadora", "JAMEF"),
        regiao_atendida_id=55,
        praca_id=10,
        praca="CAPITAL",
        modalidade_id=1,
        modalidade="RODOVIARIO",
        fator_cubagem=money(tar.get("fator_cubagem", "300")),
        uf_destino=tar.get("uf_destino", "RJ"),
        tabela_frete_id=201,
        nome_tabela="Tabela de exemplo",
        tabela_frete_peso_id=9722,
        faixa_valor_id=66597,
        peso_taxado=money(tar["peso_taxado"]),
        preco_frete_peso=money(tar["preco_frete_peso"]),
        frete_minimo=(money(tar["frete_minimo"]) if tar.get("frete_minimo") else None),
        icms=money(tar.get("icms", "0")),
        ad_valorem=(money(tar["ad_valorem"]) if tar.get("ad_valorem") else None),
        tarcli=tar.get("tarcli", "AD"),
        prazo_dias=tar.get("prazo_dias", 3),
        perfil=perfil,
    )
    return calcular(embarque, ctx, quando)


print("modos prontos: faixa e calculo")


modos prontos: faixa e calculo


## 4. Comparar esperado × motor


In [5]:
CAMPOS_CALCULO = (
    "peso_taxado_kg", "frete_peso", "generalidades", "sobre_frete",
    "icms", "frete_liquido", "frete_total",
)


def _campo(campo, esperado, motor, bateu, diferenca=None):
    item = {"campo": campo, "esperado": esperado, "motor": motor, "bateu": bateu}
    if diferenca is not None:
        item["diferenca"] = diferenca
    return item


def comparar_faixa(caso, obtido) -> list:
    esp = caso["esperado"]
    linhas = []
    if "motivo" in esp:
        bateu = (not obtido["ok"]) and obtido["motivo"] == esp["motivo"]
        linhas.append(_campo("motivo", esp["motivo"], obtido["motivo"] or "nenhum", bateu))
        return linhas
    esperado = money(esp["preco"])
    if not obtido["ok"]:
        linhas.append(_campo("preco", brl(esperado), f"descarte {obtido['motivo']}", False))
        return linhas
    obtido_preco = money(obtido["preco"])
    dif = abs(esperado - obtido_preco)
    linhas.append(_campo("preco", brl(esperado), brl(obtido_preco), dif <= tolerancia, brl(dif)))
    if "aplicou_excedente" in esp:
        linhas.append(_campo(
            "aplicou_excedente",
            "sim" if esp["aplicou_excedente"] else "nao",
            "sim" if obtido["aplicou_excedente"] else "nao",
            bool(esp["aplicou_excedente"]) is bool(obtido["aplicou_excedente"]),
        ))
    return linhas


def comparar_calculo(caso, cotacao) -> list:
    esp = caso["esperado"]
    linhas = []
    for campo in CAMPOS_CALCULO:
        if campo not in esp:
            continue
        esperado = money(esp[campo])
        obtido = money(getattr(cotacao, campo))
        dif = abs(esperado - obtido)
        linhas.append(_campo(campo, brl(esperado), brl(obtido), dif <= tolerancia, brl(dif)))

    obtidos = {c.codigo: c.valor for c in cotacao.componentes}
    previstos = esp.get("componentes") or {}
    for codigo, valor in previstos.items():
        esperado = money(valor)
        obtido = obtidos.get(codigo)
        if obtido is None:
            linhas.append(_campo(f"componente {codigo}", brl(esperado), "ausente", False))
            continue
        dif = abs(esperado - obtido)
        linhas.append(_campo(f"componente {codigo}", brl(esperado), brl(obtido), dif <= tolerancia, brl(dif)))
    for codigo in sorted(set(obtidos) - set(previstos)):
        linhas.append(_campo(f"componente {codigo}", "ausente", brl(obtidos[codigo]), False))

    if "avisos" in esp:
        esp_a = set(esp["avisos"])
        obt_a = {a.value for a in cotacao.avisos}
        linhas.append(_campo(
            "avisos",
            ", ".join(sorted(esp_a)) or "nenhum",
            ", ".join(sorted(obt_a)) or "nenhum",
            esp_a == obt_a,
        ))

    if "eventuais" in esp:
        memoria = cotacao.memoria_calculo or {}
        obt_ev = {k: money(v) for k, v in (memoria.get("eventuais") or {}).items()}
        for nome, valor in esp["eventuais"].items():
            esperado = money(valor)
            obtido = obt_ev.get(nome)
            if obtido is None:
                linhas.append(_campo(f"eventual {nome}", brl(esperado), "ausente", False))
                continue
            dif = abs(esperado - obtido)
            linhas.append(_campo(f"eventual {nome}", brl(esperado), brl(obtido), dif <= tolerancia, brl(dif)))

    if "frete_peso_aplicou_minimo" in esp:
        linha_fp = next(c for c in cotacao.componentes if c.codigo == "FRETE_PESO")
        esperado_flag = bool(esp["frete_peso_aplicou_minimo"])
        linhas.append(_campo(
            "frete_peso_aplicou_minimo",
            "sim" if esperado_flag else "nao",
            "sim" if linha_fp.aplicou_minimo else "nao",
            linha_fp.aplicou_minimo is esperado_flag,
        ))
    return linhas


def resposta_calculo(cotacao) -> dict:
    memoria = cotacao.memoria_calculo or {}
    return {
        "peso_taxado_kg": brl(cotacao.peso_taxado_kg),
        "frete_peso": brl(cotacao.frete_peso),
        "generalidades": brl(cotacao.generalidades),
        "sobre_frete": brl(cotacao.sobre_frete),
        "icms": brl(cotacao.icms),
        "frete_liquido": brl(cotacao.frete_liquido),
        "frete_total": brl(cotacao.frete_total),
        "componentes": [
            {"codigo": c.codigo, "valor": brl(c.valor), "aplicou_minimo": c.aplicou_minimo}
            for c in cotacao.componentes
        ],
        "avisos": [a.value for a in cotacao.avisos],
        "eventuais": memoria.get("eventuais") or {},
    }


print("comparadores prontos")


comparadores prontos


## 5. Rodar tudo e gravar a evidência


In [6]:
resultados = []

for caso in casos:
    modo = caso.get("modo", "calculo")
    if modo == "faixa":
        obtido = rodar_faixa(caso)
        linhas = comparar_faixa(caso, obtido)
        resposta = obtido
        total_txt = obtido.get("preco") or obtido.get("motivo") or "?"
    else:
        cotacao = rodar_calculo(caso)
        linhas = comparar_calculo(caso, cotacao)
        resposta = resposta_calculo(cotacao)
        total_txt = resposta["frete_total"]

    bateu = all(item["bateu"] for item in linhas)
    resultados.append({
        "n": caso.get("n"),
        "id": caso["id"],
        "titulo": caso["titulo"],
        "camada": caso["camada"],
        "modo": modo,
        "o_que_testa": caso["o_que_testa"],
        "peculiaridade": caso["peculiaridade"],
        "resposta_do_motor": resposta,
        "comparacao": {
            "bateu": bateu,
            "campos": linhas,
        },
        "total": total_txt,
    })
    marca = "ok  " if bateu else "FALHA"
    print(f"{marca}  caso {caso.get('n', '?'):>2}  {caso['id']:<28} → {total_txt}")
    if not bateu:
        for item in linhas:
            if not item["bateu"]:
                print(f"        {item['campo']}: esperado {item['esperado']}  motor {item['motor']}")

passaram = sum(1 for r in resultados if r["comparacao"]["bateu"])
falharam = len(resultados) - passaram
print()
print(f"{passaram} de {len(resultados)} passaram · {falharam} falharam")

SAIDA_DIR.mkdir(exist_ok=True)
relatorio = {
    "gerado_em": datetime.now().isoformat(timespec="seconds"),
    "data_referencia": quando.isoformat(),
    "resumo": {"casos": len(resultados), "passaram": passaram, "falharam": falharam},
    "mapa": [
        {
            "n": r.get("n"),
            "id": r["id"],
            "titulo": r["titulo"],
            "camada": r["camada"],
            "bateu": r["comparacao"]["bateu"],
            "resultado": r["total"],
        }
        for r in resultados
    ],
    "casos": resultados,
}
SAIDA_PATH.write_text(json.dumps(relatorio, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"evidência: {SAIDA_PATH}")


ok    caso  1  faixa_dentro_7kg             → 200.00
ok    caso  2  faixa_limite_70kg            → 161.88
ok    caso  3  excedente_acima_do_teto      → 231.48
ok    caso  4  peso_fora_sem_excedente      → PESO_FORA_DAS_FAIXAS
ok    caso  5  so_frete_peso                → 250.00
ok    caso  6  minimo_do_frete_pega         → 77.40
ok    caso  7  minimo_do_frete_nao_pega     → 250.00
ok    caso  8  gris_percentual              → 270.00
ok    caso  9  gris_piso                    → 255.00
ok    caso 10  gris_e_despacho              → 285.00
ok    caso 11  tarcli_liga                  → 309.36
ok    caso 12  tarcli_desliga               → 250.00
ok    caso 13  pallet_informado             → 309.36
ok    caso 14  pallet_estimado              → 368.72
ok    caso 15  pedagio_rj                   → 257.00
ok    caso 16  pedagio_ba                   → 250.00
ok    caso 17  pedagio_rj_101kg             → 264.00
ok    caso 18  peso_cubado                  → 250.00
ok    caso 19  icms_12           